In [ ]:
from PIL import Image
from peft import PeftModel
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from IPython.display import display, Image as IPImage
import torch

# ----------------------------
# Load model + LoRA adapter
# ----------------------------
base_model = "unsloth/Qwen3-VL-2B-Instruct"
adapter = "user55442/Vilnius-Bus-Stop-LLM"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    ignore_mismatched_sizes=True,
)
model = PeftModel.from_pretrained(model, adapter)
processor = AutoProcessor.from_pretrained(base_model)
model.eval()

# ----------------------------
# Inference helper
# ----------------------------
def run_test(image_path, prompt):
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(text=[text], images=[image], return_tensors="pt")
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
    ]

    response = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0]

    print(f"\n{'=' * 60}")
    print(f"IMAGE: {image_path}")
    print(f"PROMPT: {prompt}")
    print(f"{'=' * 60}")
    display(IPImage(filename=image_path))
    print(f"\nRESPONSE:\n{response}")

# ----------------------------
# Tests
# ----------------------------
run_test("bus_stop.jpg", "What bus stop is shown in this image?")
run_test("dog.png", "Describe this image.")